# Chapter 1.1 — Pixels, filters, and CNNs

This notebook is the “why vision models look the way they do” lab.

You will **see** each idea, not just read a definition:

1. An image is a grid of numbers.
2. DCT asks which frequencies you can throw away (JPEG-style).
3. Convolution is a sliding detector.
4. Sobel / Prewitt are hand-designed edge detectors.
5. SIFT finds repeatable points so you can match a crop to a full image.
6. CNNs *learn* those detectors; ResNet adds a skip; U-Net copies details back.

**Colab:** CPU is fine. Run top to bottom.

**What you should notice:** every later VLM still starts as “pixels → local patterns → a vector.”


## 0. Setup


In [ ]:
# Colab / local installs. Safe to re-run.
import sys
import subprocess

pkgs = [
    "numpy",
    "matplotlib",
    "opencv-python-headless",
    "pillow",
    "torch",
    "torchvision",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image

plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["image.cmap"] = "gray"
np.random.seed(0)
torch.manual_seed(0)

print("numpy", np.__version__, "| cv2", cv2.__version__, "| torch", torch.__version__)


## 1. An image is a grid of brightness values

A grayscale image is a matrix: row = y, column = x, value = how bright that pixel is.

Color is three stacked matrices (R, G, B). Models do not “see a bee.” They see numbers.


In [ ]:
# Build a tiny synthetic scene: dark background, bright disk, a sharp square.
h, w = 128, 128
yy, xx = np.mgrid[0:h, 0:w]
disk = ((xx - 40) ** 2 + (yy - 40) ** 2) < 22**2
square = (xx > 80) & (xx < 110) & (yy > 70) & (yy < 110)
img = np.zeros((h, w), dtype=np.float32)
img[disk] = 1.0
img[square] = 0.7
img += 0.04 * np.random.randn(h, w)
img = np.clip(img, 0, 1)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(img)
axes[0].set_title("Synthetic image")
axes[1].imshow(img[30:55, 30:55], interpolation="nearest")
axes[1].set_title("Zoom: individual pixels")
for ax in axes:
    ax.axis("off")
plt.show()
print("shape", img.shape, "| min", float(img.min()), "| max", float(img.max()))


**What you should notice:** the zoomed crop looks like a checker of numbers. Convolution, DCT, and neural nets all operate on this grid.

---

## 2. DCT: keep the slow waves, drop the fast ones

JPEG does not store every pixel. It splits the image into 8×8 blocks and writes each block as a mix of **cosine waves**.

- Low frequencies (top-left of the DCT block) = smooth shading.
- High frequencies (bottom-right) = sharp edges and noise.

If you zero out the high frequencies and invert the DCT, the picture still looks “like itself,” just softer. That is compression: throw away coefficients people barely see.


In [ ]:
def block_dct2(block: np.ndarray) -> np.ndarray:
    return cv2.dct(block.astype(np.float32))


def block_idct2(coeff: np.ndarray) -> np.ndarray:
    return cv2.idct(coeff.astype(np.float32))


def dct_compress(image: np.ndarray, keep: int) -> tuple[np.ndarray, np.ndarray]:
    """keep = how many leading frequencies to keep along each axis of an 8x8 block."""
    out = np.zeros_like(image)
    energy = np.zeros((8, 8), dtype=np.float32)
    n_blocks = 0
    for y in range(0, image.shape[0] - 7, 8):
        for x in range(0, image.shape[1] - 7, 8):
            block = image[y : y + 8, x : x + 8]
            coeff = block_dct2(block)
            mask = np.zeros((8, 8), dtype=np.float32)
            mask[:keep, :keep] = 1.0
            recon = block_idct2(coeff * mask)
            out[y : y + 8, x : x + 8] = recon
            energy += np.abs(coeff)
            n_blocks += 1
    return np.clip(out, 0, 1), energy / max(n_blocks, 1)


keeps = [1, 2, 4, 8]
recons = []
energies = []
for k in keeps:
    rec, eng = dct_compress(img, k)
    recons.append(rec)
    energies.append(eng)

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, rec, k in zip(axes, recons, keeps):
    ax.imshow(rec)
    ax.set_title(f"keep {k}x{k} / 8x8")
    ax.axis("off")
plt.suptitle("DCT reconstruction: more coefficients → sharper")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
im = axes[0].imshow(np.log1p(energies[-1]), cmap="magma")
axes[0].set_title("Mean |DCT| energy (log)")
axes[0].set_xlabel("u (horizontal frequency)")
axes[0].set_ylabel("v (vertical frequency)")
plt.colorbar(im, ax=axes[0], fraction=0.046)
axes[1].imshow(np.abs(recons[0] - img), cmap="magma")
axes[1].set_title("|error| when keep=1x1")
axes[1].axis("off")
plt.show()
print("Mean abs error keep=1:", float(np.mean(np.abs(recons[0] - img))))
print("Mean abs error keep=8:", float(np.mean(np.abs(recons[-1] - img))))


**What you should notice:** energy piles up in the top-left. A high `u` with low `v` is a vertical stripe pattern; low `u` with high `v` is a horizontal one. Edges suffer first when you compress.

---

## 3. Convolution: a sliding detector

Take a tiny pattern called a **kernel**. Slide it across the signal. At each stop: multiply overlapping values, add them up. That sum is one number in the **feature map**.

The kernel `[-1, 1]` asks: “did the value jump between this sample and the next?” Jumps become large positive or negative spikes. Flat regions become ~0.


In [ ]:
signal = np.array(
    [0, 0, 0, 0.2, 0.2, 0.2, 1.0, 1.0, 1.0, 0.8, 0.1, 0.1, 0.1, 0],
    dtype=np.float32,
)
kernel = np.array([-1.0, 1.0], dtype=np.float32)

# Manual 1D convolution (valid padding): same idea as the book figures.
feat = np.array(
    [signal[i] * kernel[0] + signal[i + 1] * kernel[1] for i in range(len(signal) - 1)]
)

fig, axes = plt.subplots(2, 1, figsize=(8, 4), sharex=True)
axes[0].stem(signal, basefmt=" ")
axes[0].set_title("1-D signal")
axes[1].stem(feat, linefmt="C1-", markerfmt="C1o", basefmt=" ")
axes[1].axhline(0, color="gray", lw=0.8)
axes[1].set_title("Feature map with kernel [-1, 1]")
axes[1].set_xlabel("position")
plt.tight_layout()
plt.show()
print("kernel", kernel, "| feature map", np.round(feat, 2))


Same idea in 2D. The output image is bright where the kernel “fits” the local neighborhood.

Hand-designed **edge kernels**:

- **Prewitt** and **Sobel** approximate ∂/∂x and ∂/∂y (how fast brightness changes rightward or downward).
- Sobel weights the center row/column more, so it is a bit less noisy than Prewitt.

The magnitude `hypot(gx, gy)` is “edge strength,” independent of direction.


In [ ]:
prewitt_x = np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]], dtype=np.float32)
prewitt_y = prewitt_x.T
sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float32)
sobel_y = sobel_x.T


def conv2d_np(image: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    return cv2.filter2D(image, ddepth=-1, kernel=kernel)


gx_p, gy_p = conv2d_np(img, prewitt_x), conv2d_np(img, prewitt_y)
gx_s, gy_s = conv2d_np(img, sobel_x), conv2d_np(img, sobel_y)
mag_p = np.hypot(gx_p, gy_p)
mag_s = np.hypot(gx_s, gy_s)

fig, axes = plt.subplots(2, 3, figsize=(11, 7))
titles_imgs = [
    (img, "input"),
    (gx_s, "Sobel gx (vertical edges)"),
    (gy_s, "Sobel gy (horizontal edges)"),
    (mag_s, "Sobel magnitude"),
    (mag_p, "Prewitt magnitude"),
    (np.abs(mag_s - mag_p), "|Sobel - Prewitt|"),
]
for ax, (im, title) in zip(axes.ravel(), titles_imgs):
    ax.imshow(im, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()
print("Prewitt X\n", prewitt_x)
print("Sobel X\n", sobel_x)


**What you should notice:** the disk and square boundaries light up. Vertical sides dominate `gx`; horizontal sides dominate `gy`. The feature map is not the image — it is “where this detector fired.”

---

## 4. Keypoints: find the same spot again

Classification can live on a whole-image vector. Matching and stitching need **points** that survive crop, scale, and rotation.

**SIFT** (Scale-Invariant Feature Transform) is a classic recipe:

1. Find corners / blobs that stand out at some scale.
2. Describe the local patch as a histogram of gradient directions.
3. Match descriptors between two images with nearest neighbors.

Deep models later *learn* descriptors. The job is the same: a vector that says “this is the same physical point.”


In [ ]:
# Use a real photo if the network is available; otherwise fall back to the synthetic image.
from urllib.request import urlopen

BEE_URL = "https://huggingface.co/datasets/vlmbook/images/resolve/main/bee.jpg"


def load_bgr(url: str, size=None):
    try:
        with urlopen(url, timeout=20) as f:
            arr = np.frombuffer(f.read(), dtype=np.uint8)
        bgr = cv2.imdecode(arr, cv2.IMREAD_COLOR)
        if bgr is None:
            raise RuntimeError("decode failed")
        if size is not None:
            bgr = cv2.resize(bgr, size)
        return bgr
    except Exception as exc:
        print("Could not download bee.jpg, using synthetic image:", exc)
        gray_u8 = (img * 255).astype(np.uint8)
        bgr = cv2.cvtColor(gray_u8, cv2.COLOR_GRAY2BGR)
        return cv2.resize(bgr, (320, 240))


bgr = load_bgr(BEE_URL, size=(480, 320))
gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
h, w = gray.shape
# Crop a patch from the center-right so matching is non-trivial.
crop = gray[h // 4 : 3 * h // 4, w // 3 : 4 * w // 5].copy()

sift = cv2.SIFT_create()
kp1, des1 = sift.detectAndCompute(gray, None)
kp2, des2 = sift.detectAndCompute(crop, None)
print(f"full image keypoints: {len(kp1)} | crop keypoints: {len(kp2)}")

bf = cv2.BFMatcher(cv2.NORM_L2)
matches = []
if des1 is not None and des2 is not None:
    raw = bf.knnMatch(des2, des1, k=2)
    # Lowe ratio test: keep matches that are clearly better than the 2nd neighbor.
    for pair in raw:
        if len(pair) < 2:
            continue
        m, n = pair
        if m.distance < 0.75 * n.distance:
            matches.append(m)

matches = sorted(matches, key=lambda m: m.distance)[:40]
vis_kp = cv2.drawKeypoints(
    gray, kp1[:80], None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
)
match_vis = cv2.drawMatches(
    crop,
    kp2,
    gray,
    kp1,
    matches,
    None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS,
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].imshow(cv2.cvtColor(vis_kp, cv2.COLOR_BGR2RGB))
axes[0].set_title("SIFT keypoints on full image")
axes[1].imshow(cv2.cvtColor(match_vis, cv2.COLOR_BGR2RGB))
axes[1].set_title(f"crop ↔ image matches (kept {len(matches)})")
for ax in axes:
    ax.axis("off")
plt.show()


**What you should notice:** many lines should land on the same object parts in the crop and the full photo. A few wrong matches are expected. Handcrafted descriptors struggle when lighting, style, or category shift — that is the gap CNNs and later CLIP-style models fill.

---

## 5. CNNs: the kernels become trainable

A convolutional layer is the same sliding multiply-add, but the kernel values start random and **gradient descent edits them**.

If you have 32 kernels, you get 32 feature maps: 32 different “what am I looking for?” questions asked at every location.

Task heads sit on top of a shared backbone:

| Task | Extra head | Output |
| --- | --- | --- |
| Classification | pooling + linear | one label for the whole image |
| Detection | box + class heads on a grid | “what and where” |
| Semantic segmentation | per-pixel classifier | class for every pixel |
| Instance segmentation | masks per object | overlapping objects stay separate |

**Transfer learning:** train the backbone on a huge dataset, then reuse those filters on a small dataset. You are not starting from random edges; you already have useful detectors.


In [ ]:
class TinyCNN(nn.Module):
    def __init__(self, n_classes=2):
        super().__init__()
        self.conv = nn.Conv2d(1, 8, kernel_size=3, padding=1)
        # Keep a 4x4 grid. Global pool would erase LEFT vs RIGHT.
        self.pool = nn.AdaptiveAvgPool2d(4)
        self.fc = nn.Linear(8 * 4 * 4, n_classes)

    def forward(self, x):
        feats = torch.relu(self.conv(x))
        pooled = self.pool(feats).flatten(1)
        return self.fc(pooled), feats


def make_blob_batch(n=32, size=32):
    """Class 0: blob on the left. Class 1: blob on the right."""
    imgs = torch.zeros(n, 1, size, size)
    labels = torch.randint(0, 2, (n,))
    for i, y in enumerate(labels):
        cx = 8 if y.item() == 0 else 24
        cy = 16
        yy, xx = torch.meshgrid(torch.arange(size), torch.arange(size), indexing="ij")
        blob = ((xx - cx) ** 2 + (yy - cy) ** 2) < 6**2
        imgs[i, 0][blob] = 1.0
        imgs[i, 0] += 0.05 * torch.randn(size, size)
    return imgs, labels


model = TinyCNN()
opt = torch.optim.Adam(model.parameters(), lr=0.05)
loss_hist = []
for step in range(80):
    xb, yb = make_blob_batch()
    logits, _ = model(xb)
    loss = F.cross_entropy(logits, yb)
    opt.zero_grad()
    loss.backward()
    opt.step()
    loss_hist.append(float(loss.item()))

xb, yb = make_blob_batch(n=4)
with torch.no_grad():
    logits, feats = model(xb)
    pred = logits.argmax(-1)

fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].plot(loss_hist)
axes[0].set_title("training loss")
axes[0].set_xlabel("step")
im = axes[1].imshow(model.conv.weight.detach()[0, 0].numpy(), cmap="coolwarm")
axes[1].set_title("learned kernel #0")
plt.colorbar(im, ax=axes[1], fraction=0.046)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 4, figsize=(10, 4))
for i in range(4):
    axes[0, i].imshow(xb[i, 0].numpy())
    axes[0, i].set_title(f"y={int(yb[i])} pred={int(pred[i])}")
    axes[0, i].axis("off")
    axes[1, i].imshow(feats[i, 0].numpy())
    axes[1, i].set_title("feat map 0")
    axes[1, i].axis("off")
plt.suptitle("Input vs first feature map after training")
plt.tight_layout()
plt.show()
print("final loss", loss_hist[-1], "| batch acc", float((pred == yb).float().mean()))


**What you should notice:** loss falls because the 3×3 kernels stop being noise and start firing on left vs right blobs. That is the same story as Sobel, except nobody typed the numbers in.

---

## 6. ResNet skip: add the input back

Deep stacks of layers can wash out the original signal (vanishing gradients). A **residual block** computes a residual `F(x)` and returns `x + F(x)`.

The layer is allowed to say “change nothing” by driving `F(x)` toward 0. Batch normalization keeps hidden values from drifting; ReLU (`max(0, z)`) is the usual nonlinearity.

A **bottleneck** residual (1×1 compress → 3×3 → 1×1 expand) is just a cheaper `F`.


In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels=8):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        f = torch.relu(self.bn1(self.conv1(x)))
        f = self.bn2(self.conv2(f))
        return torch.relu(x + f)


x = torch.randn(1, 8, 16, 16)
block = ResidualBlock()
with torch.no_grad():
    f_only = block.bn2(block.conv2(torch.relu(block.bn1(block.conv1(x)))))
    out = block(x)

print("||x||", float(x.norm()))
print("||F(x)||  (the residual)", float(f_only.norm()))
print("||x + F(x)|| after ReLU", float(out.norm()))
print("mean |out - x|", float((out - x).abs().mean()), "← skip kept a copy of x")

# Bottleneck shapes: 64 → 16 → 16 → 64
bottleneck = nn.Sequential(
    nn.Conv2d(64, 16, 1),
    nn.ReLU(),
    nn.Conv2d(16, 16, 3, padding=1),
    nn.ReLU(),
    nn.Conv2d(16, 64, 1),
)
dummy = torch.zeros(1, 64, 8, 8)
print("bottleneck params", sum(p.numel() for p in bottleneck.parameters()))
print("plain 3x3 64→64 params", sum(p.numel() for p in nn.Conv2d(64, 64, 3, padding=1).parameters()))
_ = bottleneck(dummy)


**What you should notice:** `x + F(x)` cannot forget `x`. That is the whole trick.

---

## 7. U-Net: downsample for context, upsample, then copy the details back

Segmentation wants a label **per pixel**. Pooling gives context but wrecks spatial precision.

U-Net is an encoder–decoder:

- Encoder: conv + downsample (what is this region?).
- Decoder: upsample (paint a full-resolution map).
- **Skip concatenations:** stitch encoder feature maps onto the decoder so edges survive.

Medical imaging popularized it because you can get sharp masks from small datasets.


In [ ]:
class MiniUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = nn.Conv2d(1, 8, 3, padding=1)
        self.enc2 = nn.Conv2d(8, 16, 3, padding=1)
        self.dec = nn.Conv2d(16 + 8, 8, 3, padding=1)
        self.head = nn.Conv2d(8, 1, 1)

    def forward(self, x):
        e1 = torch.relu(self.enc1(x))              # full resolution
        e2 = torch.relu(self.enc2(F.max_pool2d(e1, 2)))
        up = F.interpolate(e2, scale_factor=2, mode="nearest")
        cat = torch.cat([up, e1], dim=1)           # skip connection (concat, not add)
        d = torch.relu(self.dec(cat))
        return self.head(d), e1, e2, cat


unet = MiniUNet()
# Toy mask: the square from our synthetic image.
mask = torch.tensor(img)[None, None]
mask_bin = (mask > 0.5).float()
opt = torch.optim.Adam(unet.parameters(), lr=0.03)
losses = []
for _ in range(60):
    logit, *_ = unet(mask)
    loss = F.binary_cross_entropy_with_logits(logit, mask_bin)
    opt.zero_grad()
    loss.backward()
    opt.step()
    losses.append(float(loss.item()))

with torch.no_grad():
    logit, e1, e2, cat = unet(mask)
    pred = torch.sigmoid(logit)[0, 0].numpy()

print("skip concat channels", cat.shape[1], "(decoder 16 + encoder 8)")
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
axes[0].imshow(img)
axes[0].set_title("input")
axes[1].plot(losses)
axes[1].set_title("mask BCE loss")
axes[2].imshow(pred, cmap="magma")
axes[2].set_title("predicted mask")
for ax in (axes[0], axes[2]):
    ax.axis("off")
plt.tight_layout()
plt.show()


**What you should notice:** the skip tensor has more channels because it **concatenates** encoder details with decoder context. ResNet **adds**; U-Net **concatenates**. Both are “don’t lose the earlier representation.”

---

## Pause — connect this to VLMs

| Classical idea | Neural version | VLM version |
| --- | --- | --- |
| DCT / local frequencies | conv filters | patch embeddings |
| Sobel edges | learned conv1 | ViT still cares about local structure |
| SIFT descriptor | CNN pooled vector | CLIP image tower |
| Task-specific head | classify / detect / segment | text as a flexible head |

Next notebook: language models, Vision Transformers, and CLIP — the moment image vectors and text vectors share a space.
